# 01 — LangChain 基礎：用「生產線」比喻 Runnable 與 LCEL

**這份要學什麼**
- `Runnable` 介面：`.invoke()` / `.stream()` / `.batch()`
- LCEL：用 `|` 把 prompt、模型、parser 串成一條生產線
- 為什麼「一條直線」的 chain 撐不住分支、迴圈——帶出後面為什麼需要 LangGraph

在比較 LangChain / LangGraph 的 agent 寫法之前，先搞懂 LangChain 最核心的概念：**`Runnable`**。之後你會發現 LangGraph 的每個「節點」其實也是「丟東西進去、吐東西出來」的可呼叫單元，概念是相通的——LangGraph 只是把「一條直線的鏈」換成「一張圖」，讓你能表達分支、迴圈、狀態，這些「一條直線」表達不出來的東西。

## 為什麼需要 LCEL（LangChain Expression Language）

想像一條**生產線**：原料從這頭進去，經過一站一站的加工，成品從那頭出來。

```
輸入 ──▶ [ Prompt 模板 ] ──▶ [ 呼叫 LLM ] ──▶ [ 輸出解析器 ] ──▶ 輸出
           把變數套進句子       模型生成回答       只取出純文字
```

沒有 LCEL 之前，串接「套模板 → 呼叫模型 → 解析輸出」要手動呼叫三次、自己接手把上一步的結果傳給下一步。LCEL 用 `|`（pipe，就像水管一節接一節）把這件事變成一行：`prompt | llm | parser`。能這樣接，是因為每個組件都遵守同一種介面規格——`Runnable`（都有 `.invoke()` 呼叫一次、`.stream()` 逐字輸出、`.batch()` 一次處理多筆，以及對應的非同步版本），所以可以像水管零件一樣任意組合。

In [1]:
import sys

sys.path.insert(0, ".")
from _llm import get_llm, has_api_key, scripted_model

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

## 生產線上的三個零件

- **`ChatPromptTemplate`**：把你填的變數套進事先寫好的訊息模板，輸出一個 `PromptValue`（可以再轉成訊息列表，餵給模型）
- **`BaseChatModel`**（這裡是 `get_llm()` 回傳的實例）：接收訊息，輸出模型回覆的 `AIMessage`
- **`StrOutputParser`**：把 `AIMessage` 轉成單純的字串（只取裡面的 `.content`）

這三個零件都是 `Runnable`，接口一致，才能用 `|` 直接串成一條 chain。

In [2]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一位{role}，回答控制在兩句話以內。"),
        ("human", "{question}"),
    ]
)

print(prompt.input_variables)
print(prompt.invoke({"role": "資深 Python 工程師", "question": "什麼是 Runnable？"}))

['question', 'role']
messages=[SystemMessage(content='你是一位資深 Python 工程師，回答控制在兩句話以內。', additional_kwargs={}, response_metadata={}), HumanMessage(content='什麼是 Runnable？', additional_kwargs={}, response_metadata={})]


上面這格只是在組裝 `PromptValue`（生產線的第一站），還沒真的呼叫 LLM，所以不需要 API key。

接下來把三個 `Runnable` 用 `|` 接成完整的一條生產線——用 `scripted_model()` 當第二站，這樣就算沒有 API key，也能跑出真正的執行結果（不是模擬，是這個假模型真的被呼叫、真的回傳資料）。

In [3]:
chain = prompt | scripted_model(["Runnable 是 LangChain 裡所有可呼叫元件共用的介面。"]) | StrOutputParser()

result: str = chain.invoke(
    {"role": "資深 Python 工程師", "question": "什麼是 Runnable？"}
)
print(result)

Runnable 是 LangChain 裡所有可呼叫元件共用的介面。


如果你有 API key，同樣的組合換成 `get_llm()`：

In [4]:
if has_api_key():
    real_chain = prompt | get_llm() | StrOutputParser()
    print(real_chain.invoke({"role": "資深 Python 工程師", "question": "什麼是 Runnable？"}))
else:
    print("尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面 scripted_model 已經展示了完整流程）。")

Runnable 是一種可被執行的抽象介面，通常代表能接收輸入並產生輸出的元件。  
在 LangChain 中，Runnable 支援 `invoke`、`batch`、`stream` 等操作，並可串接成處理流程。


## `.stream()`：LCEL 附贈的能力

因為生產線上每個零件都遵守同一套 `Runnable` 介面，只要底層模型支援「逐字吐出結果」（streaming），整條 chain 完全不用改寫，就能用 `.stream()` 一個字一個字印出來，而不是等全部生成完才一次顯示。這個能力在 LangGraph 裡也會用到（`stream_mode="messages"`），是同一套概念。

`scripted_model` 的 `.stream()` 會把劇本文字依空白拆成好幾個小塊，模擬「逐字輸出」的效果——下面印出來一格一格斷開的樣子就是這個原因。

In [5]:
stream_chain = prompt | scripted_model(["LCEL 讓你用 pipe 運算子 串接 多個 Runnable。"]) | StrOutputParser()

for chunk in stream_chain.stream({"role": "資深 Python 工程師", "question": "什麼是 LCEL？"}):
    print(chunk, end="", flush=True)
print()

LCEL

讓你用

pipe

運算子

串接

多個

Runnable。

如果你有 API key，同樣可以直接對真模型呼叫 `.stream()`：

In [6]:
if has_api_key():
    real_stream_chain = prompt | get_llm() | StrOutputParser()
    for chunk in real_stream_chain.stream({"role": "資深 Python 工程師", "question": "什麼是 LCEL？"}):
        print(chunk, end="", flush=True)
    print()
else:
    print("尚未設定 OPENAI_API_KEY，跳過真模型 streaming（上面已經展示了 .stream() 的用法）。")

LC

EL

（

Lang

Chain

...

重

試

等

功能

。

## 小結：LCEL 的極限在哪裡

生產線（chain）很適合「A 站接 B 站接 C 站」這種固定順序的流程。但一旦你需要：

- 依模型的回答**決定接下來走哪條路**（分支）
- 讓模型自己**決定要不要重複呼叫工具**，而且呼叫幾次都不確定（迴圈，例如 agent 的 tool-calling 迴圈）
- 在**好幾輪對話之間**保留之前講過的內容（記憶）

生產線這種「一條直線」的比喻就撐不住了，得用巢狀的 `RunnableBranch`、自訂函式硬湊，程式碼會變得很難讀。

這正是下一份 notebook（`02_three_agents_compare`）要處理的問題——而 LangGraph 的 `StateGraph` 就是為了直接表達「圖」（有分支、有迴圈）而生的解法，不用再硬湊。